In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# pip install kagglehub[pandas-datasets]

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

kenpom = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS, 
    "nishaanamin/march-madness-data", 
    "KenPom Barttorvik.csv",
    pandas_kwargs={"usecols": ["YEAR", "TEAM", "KADJ EM", "BARTHAG", "SEED", "ROUND", "QUAD ID"]}
)
heat = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "nishaanamin/march-madness-data",
    "Heat Check Ratings.csv",
    pandas_kwargs={"usecols": ["YEAR", "TEAM", "EASY DRAW", "TOUGH DRAW", "DARK HORSE", "UPSET ALERT", "CINDERELLA"]}
)
shoot = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS, 
    "nishaanamin/march-madness-data", 
    "Shooting Splits.csv",
    pandas_kwargs={"usecols": ["YEAR", "TEAM", "THREES FG%", "THREES FG%D", "THREES SHARE", "CLOSE TWOS FG%", "CLOSE TWOS FG%D", "DUNKS FG%"]}
)

print("\nKenPom Barttorvik\n", kenpom.head())
print("\nHeat Check Ratings\n", heat.head())
print("\nShooting Splits\n", shoot.head())

100%|██████████| 545k/545k [00:00<00:00, 1.34MB/s]


100%|██████████| 17.4k/17.4k [00:00<00:00, 1.09MB/s]


100%|██████████| 170k/170k [00:00<00:00, 643kB/s]


KenPom Barttorvik
    YEAR  QUAD ID      TEAM  SEED  ROUND  KADJ EM  BARTHAG
0  2026        1     Akron    12      0  12.7986    0.773
1  2026        1   Alabama     4      0  25.7196    0.934
2  2026        2   Arizona     1      0  37.6556    0.978
3  2026        2  Arkansas     4      0  26.0527    0.934
4  2026        2       BYU     6      0  23.2459    0.887

Heat Check Ratings
    YEAR         TEAM  EASY DRAW  TOUGH DRAW  DARK HORSE  UPSET ALERT  \
0  2026   Louisville      False       False        True        False   
1  2026    Tennessee      False       False        True        False   
2  2025  Connecticut      False       False        True        False   
3  2025     Oklahoma      False        True       False        False   
4  2025      Memphis      False        True       False         True   

   CINDERELLA  
0       False  
1       False  
2       False  
3       False  
4       False  

Shooting Splits
    YEAR      TEAM  DUNKS FG%  CLOSE TWOS FG%  CLOSE TWOS FG%D  T

In [5]:
df = kenpom.merge(heat, on=["YEAR", "TEAM"], how="inner").merge(shoot, on=["YEAR", "TEAM"], how="inner")
df.head()

,YEAR,QUAD ID,TEAM,SEED,ROUND,KADJ EM,BARTHAG,EASY DRAW,TOUGH DRAW,DARK HORSE,UPSET ALERT,CINDERELLA,DUNKS FG%,CLOSE TWOS FG%,CLOSE TWOS FG%D,THREES FG%,THREES SHARE,THREES FG%D
0,2026,4,Louisville,6,0,25.4242,0.936,False,False,True,False,False,88.1,64.7,60.3,35.7,52.8,32.7
1,2026,1,Tennessee,6,0,26.0249,0.941,False,False,True,False,False,89.3,60.8,56.9,33.4,31.7,30.6
2,2025,4,Alabama St.,16,64,-9.2350,0.270,False,True,False,False,False,77.0,53.1,57.7,32.9,42.9,33.4
3,2025,3,Arkansas,10,16,17.6925,0.862,False,True,False,False,False,94.5,64.6,55.7,33.3,36.7,31.9
4,2025,2,Baylor,9,32,21.1589,0.903,False,False,True,False,False,95.6,62.8,68.6,34.7,40.2,35.4


In [6]:
# Alternative target: did the team win at least one game? (beat R64?)
# This turns it into binary classification which is much more learnable
# and more directly useful for bracket simulation

df_past = df[df["ROUND"] != 0].copy()
df_2026 = df[df["ROUND"] == 0].copy()

rounds = {64: 1, 32: 2, 16: 3, 8: 4, 4: 5, 2: 6, 1: 7}
df_past["ROUND_ENCODED"] = df_past["ROUND"].map(rounds)

before = len(df_past)
df_past = df_past.dropna(subset=["ROUND_ENCODED"]).copy()
df_past["ROUND_ENCODED"] = df_past["ROUND_ENCODED"].astype(int)
print(f"Dropped {before - len(df_past)} rows with unmapped ROUND values")
print("Class distribution:\n", df_past["ROUND_ENCODED"].value_counts().sort_index())

Dropped 2 rows with unmapped ROUND values
Class distribution:
 ROUND_ENCODED
1    145
2     88
3     49
4     23
5      9
6      4
7      4
Name: count, dtype: int64


In [21]:
X = df_past.drop(columns=["ROUND_ENCODED", "TEAM", "QUAD ID"])
y = df_past["ROUND_ENCODED"]

X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 322 entries, 2 to 325
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   YEAR             322 non-null    int64  
 1   SEED             322 non-null    int64  
 2   ROUND            322 non-null    int64  
 3   KADJ EM          322 non-null    float64
 4   BARTHAG          322 non-null    float64
 5   EASY DRAW        322 non-null    bool   
 6   TOUGH DRAW       322 non-null    bool   
 7   DARK HORSE       322 non-null    bool   
 8   UPSET ALERT      322 non-null    bool   
 9   CINDERELLA       322 non-null    bool   
 10  DUNKS FG%        322 non-null    float64
 11  CLOSE TWOS FG%   322 non-null    float64
 12  CLOSE TWOS FG%D  322 non-null    float64
 13  THREES FG%       322 non-null    float64
 14  THREES SHARE     322 non-null    float64
 15  THREES FG%D      322 non-null    float64
dtypes: bool(5), float64(8), int64(3)
memory usage: 31.8 KB


In [25]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
ct = ColumnTransformer([("scaling", StandardScaler(), ["SEED", "KADJ EM", "BARTHAG", "DUNKS FG%", "CLOSE TWOS FG%", "CLOSE TWOS FG%D", "THREES FG%", "THREES SHARE", "THREES FG%D"])], remainder="passthrough")

In [26]:
models = {"Logistic Regression": Pipeline([("preprocessor", ct), ("classifier", LogisticRegression(max_iter=1000, C=1.0, multi_class="multinomial", solver="lbfgs", class_weight="balanced", random_state=42))]),
    "Random Forest": Pipeline([("preprocessor", ct), ("classifier", RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_split=5, class_weight="balanced", random_state=42, n_jobs=-1))]),
    "Gradient Boosting": Pipeline([("preprocessor", ct), ("classifier", GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42))])
}

In [30]:
ROUND_LABELS = ["R64", "R32", "S16", "E8", "F4", "Final", "Champ"]
results = {}
best_score = 0
best_model = None

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cv = cross_val_score(model, X, y, cv=5, scoring="accuracy")

    if cv.mean() > best_score:
        best_model = model

    results[name] = {
        "model": model,
        "y_pred": y_pred,
        "test_acc": acc,
        "cv_mean": cv.mean(),
        "cv_std": cv.std(),
    }

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"  Test Accuracy : {acc:.4f}")
    print(f"  CV Accuracy   : {cv.mean():.4f} ± {cv.std():.4f}")
    print(f"{'='*50}")
    present_labels = sorted(y_test.unique())
    present_names = [ROUND_LABELS[i - 1] for i in present_labels]
    print(classification_report(y_test, y_pred, labels=present_labels,
                                target_names=present_names, zero_division=0))


  Logistic Regression
  Test Accuracy : 0.9385
  CV Accuracy   : 0.9689 ± 0.0140
              precision    recall  f1-score   support

         R64       1.00      1.00      1.00        29
         R32       1.00      1.00      1.00        18
         S16       1.00      1.00      1.00        10
          E8       1.00      1.00      1.00         4
          F4       0.00      0.00      0.00         2
       Final       0.00      0.00      0.00         1
       Champ       0.00      0.00      0.00         1

    accuracy                           0.94        65
   macro avg       0.57      0.57      0.57        65
weighted avg       0.94      0.94      0.94        65


  Random Forest
  Test Accuracy : 0.9846
  CV Accuracy   : 0.9626 ± 0.0211
              precision    recall  f1-score   support

         R64       1.00      1.00      1.00        29
         R32       1.00      1.00      1.00        18
         S16       1.00      1.00      1.00        10
          E8       1.00     

In [32]:
df_2026.info()

#y_pred_2026 = best_model.predict(X_2026)

<class 'pandas.core.frame.DataFrame'>
Index: 2 entries, 0 to 1
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   YEAR             2 non-null      int64  
 1   QUAD ID          2 non-null      int64  
 2   TEAM             2 non-null      object 
 3   SEED             2 non-null      int64  
 4   ROUND            2 non-null      int64  
 5   KADJ EM          2 non-null      float64
 6   BARTHAG          2 non-null      float64
 7   EASY DRAW        2 non-null      bool   
 8   TOUGH DRAW       2 non-null      bool   
 9   DARK HORSE       2 non-null      bool   
 10  UPSET ALERT      2 non-null      bool   
 11  CINDERELLA       2 non-null      bool   
 12  DUNKS FG%        2 non-null      float64
 13  CLOSE TWOS FG%   2 non-null      float64
 14  CLOSE TWOS FG%D  2 non-null      float64
 15  THREES FG%       2 non-null      float64
 16  THREES SHARE     2 non-null      float64
 17  THREES FG%D      2 non-nu